> **교육 목표** 3일차에 손으로 한 작업을 AutoML(PyCaret)로 다시 풀어, 속도와 성능을 비교합니다.
>
> **핵심 내용** `setup`(시간 Test·누수 제외·불균형) → 모델 비교 → 튜닝 → 블렌딩·스태킹 → Test 비교 → 변수 중요도 → 모델 저장

In [ ]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D6_04 PyCaret으로 3일차 문제 다시 풀기

> **오늘 질문:** 3일차에 손으로 한 작업을 AutoML은 얼마나 빨리, 얼마나 잘 해낼까?

| 3일차에 손으로 한 것 | PyCaret에서 |
|---|---|
| 시간 순 마지막 20% Test | `setup(test_data=...)` |
| 누수 컬럼 빼기 | `setup(ignore_features=...)` |
| 불균형 리샘플링 | `setup(fix_imbalance=True)` |
| 교차검증·튜닝 | `fold_strategy`, `tune_model()` |
| 모델 비교·앙상블 | `compare_models()`, `blend_models()`, `stack_models()` |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: Data Mart (누수 컬럼 포함) 📋

일부러 고장 유형 컬럼(`twf` ~ `rnf`)을 데이터에 남겨 둡니다. setup에서 빼는 연습을 하기 위해서입니다.

In [ ]:
import pandas as pd
from pycaret.classification import *

df = pd.read_parquet("../../data/factory_table.parquet")
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"]
df["strain"] = df["tool_wear"] * df["torque"]
df["grade_num"] = df["grade"].map({"L": 0, "M": 1, "H": 2})
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "vibration", "temp_diff", "power", "strain", "grade_num"]
leak = ["twf", "hdf", "pwf", "osf", "rnf"]

data = df[features + leak + ["failure"]]
cut = int(len(data) * 0.8)
train, test = data.iloc[:cut], data.iloc[cut:]       # 시간 순 80 / 20
print(f"Train {len(train)} / Test {len(test)}")

## 1. setup: 원칙을 한 줄에 ✍️

| 옵션 | 오늘의 설정 |
|---|---|
| `test_data` | 시간 순 마지막 20% |
| `fold_strategy="timeseries"`, `fold=3` | 교차검증도 시간 순서대로 |
| `data_split_shuffle=False` | 섞지 않기 (`timeseries`를 쓰려면 꼭 필요) |
| `ignore_features` | 누수 컬럼 |
| `fix_imbalance=True` | 불균형 자동 처리 (SMOTE) |
| `session_id=42` | 같은 결과 재현 |

In [ ]:
# 힌트: setup(data=..., test_data=..., target="failure", ...)

## 2. 모델 비교 ✍️

후보를 6개로 줄이고 **F1 기준**으로 정렬합니다. 상위 3개를 받아 둡니다. (20초 정도)

> 변수 이름: `top3`, `best`

In [ ]:
# 힌트: compare_models(include=[...], sort="F1", n_select=3)

## 3. 튜닝 ✍️

F1이 오르도록 하이퍼파라미터를 무작위로 5번 탐색합니다. 튜닝 결과가 더 나쁘면 PyCaret은 원래 모델을 돌려줍니다. (1분 정도)

> 변수 이름: `tuned`

In [ ]:
# 힌트: tune_model(모델, optimize="F1", n_iter=5)

## 4. 앙상블: 블렌딩과 스태킹 ✍️

상위 3개 모델을 섞어 봅니다. (스태킹은 40초 정도)

> 변수 이름: `blend`, `stack`

In [ ]:
# 힌트: blend_models(모델목록), stack_models(모델목록)

## 5. Test(마지막 20%)에서 비교 📋

`predict_model`은 setup에서 지정한 Test 데이터로 평가하고, `pull()`로 그 점수표를 가져옵니다.

In [ ]:
rows = []
for name, model in {"1위 단일": best, "튜닝": tuned, "블렌딩": blend, "스태킹": stack}.items():
    predict_model(model, verbose=False)
    r = pull().iloc[0]
    rows.append({"모델": name, "정밀도": r["Prec."], "재현율": r["Recall"], "F1": r["F1"]})
result = pd.DataFrame(rows)
result

## 6. 변수 중요도 ✍️

In [ ]:
# 힌트: plot_model(모델, plot="feature")

## 7. 최종 모델 저장 ✍️

`finalize_model`은 Train과 Test를 합쳐 다시 학습합니다. 성능 확인이 끝난 **마지막에만** 씁니다. 전처리(SMOTE 등)까지 Pipeline째 저장됩니다.

In [ ]:
# 힌트: finalize_model(모델), save_model(모델, "파일이름")

## 정리 💬

**Q1.** 3일차에 손으로 만든 모델(튜닝한 랜덤포레스트, Test F1 약 0.81)과 PyCaret 결과를 비교해 보세요. 걸린 시간과 성능은?

> 답:


**Q2.** setup에서 `ignore_features=leak`를 빼고 다시 실행하면 어떻게 될까요? `test_data` 대신 기본값(무작위 분할)을 쓰면?

> 답:
